# Novelty signal annotation results

Put the returned `answers.csv` next to `key.csv` in `output/signal_annotation/` first. A Jupyter kernel on macOS can't read `~/Downloads`.

In [ ]:
import json
import textwrap
from pathlib import Path

import pandas as pd
import yaml
from sklearn.metrics import cohen_kappa_score

DIR = Path("../../../output/signal_annotation")
DATA = Path("../../../data/human-only/pointwise.yaml")
# Per-review novelty signals, used to show the sampled reviewer's other signals.
REVIEWS = Path("./benchmark_data/iclr_data/20260313_161835/iclr_2026_top_bottom_10_percent.json")

answers = pd.read_csv(DIR / "answers.csv", dtype=str).fillna("")
df = pd.read_csv(DIR / "key.csv", dtype=str).merge(answers, on="item_id", validate="one_to_one")
df["correct"] = df["polarity"] == df["model_polarity"]
print(f"{len(df)} items, {(df['polarity'] != '').sum()} annotated")

## Extraction precision

Share of signals where the annotator's polarity matches the model's.

In [ ]:
precision = df.groupby("model_polarity")["correct"].mean()
precision["all"] = df["correct"].mean()
precision.round(3)

## Human–model agreement

Cohen's κ on polarity, with `not_novelty` as a third class.

In [ ]:
print(f"kappa = {cohen_kappa_score(df['model_polarity'], df['polarity']):.3f}")
pd.crosstab(df["model_polarity"], df["polarity"])

## Missing from the abstract

Correctly extracted signals whose judged contribution is missing from the abstract: count (`missing`), out of `n`, and `share`.

In [ ]:
correct = df[df["correct"]]
is_missing = correct["missing_from_abstract"] == "yes"
groups = (is_missing.groupby(correct["model_polarity"]), is_missing.groupby(lambda _: "all"))
missing = pd.concat([g.agg(missing="sum", n="count", share="mean") for g in groups])
missing.round(3)

## Items to review

Disagreements, signals missing from the abstract, and items with notes.

In [ ]:
ideas = yaml.safe_load(DATA.read_text())

# Each extracted signal -> all novelty signals of the review it came from.
review_signals = {}
for area in json.loads(REVIEWS.read_text()).values():
    for paper in area["top_papers"] + area["bottom_papers"]:
        for paper_review in paper["reviews"]:
            signals = [("positive", s) for s in paper_review["positive_novelty_signals"]]
            signals += [("negative", s) for s in paper_review["negative_novelty_signals"]]
            for _, s in signals:
                review_signals[s] = signals


def wrap(text):
    return textwrap.fill(text, width=100, initial_indent="    ", subsequent_indent="    ")


review = df[~df["correct"] | (df["missing_from_abstract"] == "yes") | (df["notes"] != "")]
for _, row in review.iterrows():
    idea = ideas[row["idea_id"]]
    print("=" * 104)
    print(f"{row['item_id']}  {idea['metadata']['title']}")
    print(f"model: {row['model_polarity']}   human: {row['polarity']}   "
          f"missing from abstract: {row['missing_from_abstract'] or '-'}")
    print(f"\n  SIGNAL\n{wrap(row['signal'])}")
    if not row["correct"]:
        print("\n  OTHER SIGNALS BY THE SAME REVIEWER")
        others = [(k, s) for k, s in review_signals[row["signal"]] if s != row["signal"]]
        for polarity, signal in others:
            print(wrap(f"[{polarity}] {signal}"))
        if not others:
            print("    (none)")
    print(f"\n  ABSTRACT\n{wrap(idea['idea'])}")
    if row["notes"]:
        print(f"\n  NOTES\n{wrap(row['notes'])}")
    print()